In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
    REPO_PATH = "/content/packt-modern-time-series"

    if not os.path.exists(REPO_PATH):
        os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

    os.chdir(f"{REPO_PATH}/instructor_notebooks")

    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)

print(f"✓ Setup complete — {os.getcwd()}")

---
# Module 2 — Problem Framing & Config
**Type:** [Code With Me]  
**Time:** 20 minutes  
**Job:** Lock every constant the workshop depends on. Define pooled scoring mechanics. Connect the evaluation standard to an inventory decision.



## 2.1 — Path Setup
**[Watch Only]**

---

In [14]:
import sys
from pathlib import Path

# Add project root to path — required for config.py and src/ imports
PROJECT_ROOT = Path().resolve().parent  # notebooks live one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\tacke\Documents\packt-modern-time-series


---
## 2.2 — Import Config
**[Watch Only]**

---

In [15]:
import pickle
import pandas as pd
import numpy as np

import config
from config import (
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    RANDOM_SEED,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    INTERVAL_COVERAGE,
    ARTIFACT_DIR,
    WORKSHOP_SUBSET_PATH,
)

print("Workshop configuration")
print("=" * 40)
print(f"  Forecast horizon    : {HORIZON} days")
print(f"  Season length       : {SEASON_LENGTH} days (weekly)")
print(f"  CV windows          : {N_WINDOWS}")
print(f"  Step size           : {STEP_SIZE} days")
print(f"  Refit between windows: {REFIT}")
print(f"  Random seed         : {RANDOM_SEED}")
print(f"  Live subset (micro) : {MICRO_SUBSET_N} series")
print(f"  Full subset         : {WORKSHOP_SUBSET_N} series")
print(f"  Interval coverage   : {int(INTERVAL_COVERAGE * 100)}%")

Workshop configuration
  Forecast horizon    : 28 days
  Season length       : 7 days (weekly)
  CV windows          : 3
  Step size           : 28 days
  Refit between windows: False
  Random seed         : 42
  Live subset (micro) : 50 series
  Full subset         : 1000 series
  Interval coverage   : 80%


---
## 2.3 — Why These Parameters?
**[Watch Only]**



These are not arbitrary. Each parameter maps to a production constraint.

**Horizon = 28 days**  
Standard replenishment cycle for Walmart-scale retail. Your procurement team orders 4 weeks out. A 7-day horizon is operationally useless for inventory planning at that lead time.

**Season length = 7 days**  
Weekly demand cycles dominate retail. Friday ≠ Monday. Any model that does not capture this will have structurally biased errors.

**N_WINDOWS = 3, STEP_SIZE = 28**  
Three non-overlapping 28-day evaluation windows. This gives us 84 days of out-of-sample signal across 1,000 series — enough to be statistically meaningful, not so much that it consumes the training history.

**REFIT = False**  
We train once and roll the evaluation window forward. In production, daily retraining is expensive. Testing with `REFIT=True` would inflate model performance beyond what you'd see in a real deployment.

**MICRO_SUBSET_N = 50**  
Live cells run on 50 series to stay inside the 90-second ceiling. Precomputed artifacts cover all 1,000. The patterns are identical — scale is the only difference.

---
## 2.4 — Define the CV Parameters
**[Code With Me — 2 lines]**

Fill in `n_windows` and `step_size` using the config imports above. Do not hardcode numbers.

---

In [16]:
# Cross-validation parameters for StatsForecast.
# Import the values from config — do not hardcode them here.

cv_params = {
    "h":         HORIZON,
    "n_windows": N_WINDOWS,    # __FILL_IN__: import from config
    "step_size": STEP_SIZE,    # __FILL_IN__: import from config
    "refit":     REFIT,
}

print("CV parameters locked:")
for k, v in cv_params.items():
    print(f"  {k:<12}: {v}")

CV parameters locked:
  h           : 28
  n_windows   : 3
  step_size   : 28
  refit       : False


**Expected output:**
```
CV parameters locked:
  h           : 28
  n_windows   : 3
  step_size   : 28
  refit       : False
```

---
## 2.5 — Evaluation Logic: wMAPE
**[Watch Only]**


**wMAPE — Weighted Mean Absolute Percentage Error**

$$\text{wMAPE} = \frac{\sum |y_t - \hat{y}_t|}{\sum |y_t|}$$



---
## 2.6 — Evaluation Logic: Interval Score
**[Watch Only]**

**Interval Score (Winkler Score) at 80%**

$$\text{IS}_{\alpha} = (\hat{u} - \hat{l}) + \frac{2}{\alpha}(\hat{l} - y)^+ + \frac{2}{\alpha}(y - \hat{u})^+$$

Where $\hat{l}$, $\hat{u}$ are the lower and upper bounds of the 80% interval, $\alpha = 0.20$, and $(x)^+ = \max(x, 0)$.

---
## 2.6b — Evaluation Logic: Bias
**[Watch Only]**

**Bias — Pooled Forecast Bias**

$$\text{Bias} = \frac{\sum (\hat{y}_t - y_t)}{\sum y_t}$$

Positive = systematic over-forecast. Negative = systematic under-forecast. A model with good wMAPE but large bias is making consistently directional errors — dangerous in inventory planning where over-forecasting drives excess stock and under-forecasting drives stockouts.

---
## 2.7 — Verify the Scoring Functions
**[Code With Me — 3 lines]**

Call `pooled_wmape`, `pooled_bias`, and `pooled_interval_score` on the toy arrays below. Fill in the function arguments.

---

In [17]:
from src.evaluation import pooled_wmape, pooled_bias, pooled_interval_score

# Toy arrays — verify the functions return sensible numbers
# y_hat is deliberately slightly over-forecast to show non-zero bias
y_true = np.array([100.0, 200.0, 150.0,  80.0, 120.0])
y_hat  = np.array([115.0, 180.0, 160.0,  75.0, 130.0])
lo     = np.array([ 80.0, 150.0, 120.0,  50.0,  90.0])
hi     = np.array([140.0, 220.0, 200.0, 110.0, 160.0])

wmape  = pooled_wmape(y_true, y_hat)             # __FILL_IN__: call pooled_wmape with y_true and y_hat
bias   = pooled_bias(y_true, y_hat)              # __FILL_IN__: call pooled_bias with y_true and y_hat
iscore = pooled_interval_score(y_true, lo, hi)   # __FILL_IN__: call pooled_interval_score with y_true, lo, hi

print(f"wMAPE         : {wmape:.4f}  ({wmape*100:.2f}%)")
print(f"Bias          : {bias:+.4f}  ({'over' if bias > 0 else 'under'}-forecast)")
print(f"Interval Score: {iscore:.4f}")
print()
print("All three functions are working. Importing src.evaluation is confirmed.")

wMAPE         : 0.0923  (9.23%)
Bias          : +0.0154  (over-forecast)
Interval Score: 68.0000

All three functions are working. Importing src.evaluation is confirmed.


**Expected output:**
```
wMAPE         : 0.0923  (9.23%)
Bias          : +0.0154  (over-forecast)
Interval Score: 68.0000

All three functions are working. Importing src.evaluation is confirmed.
```

---
## 2.8 — Save the Config Artifact
**[Watch Only]**

---

In [18]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

global_config = {
    "HORIZON":           HORIZON,
    "SEASON_LENGTH":     SEASON_LENGTH,
    "N_WINDOWS":         N_WINDOWS,
    "STEP_SIZE":         STEP_SIZE,
    "REFIT":             REFIT,
    "RANDOM_SEED":       RANDOM_SEED,
    "MICRO_SUBSET_N":    MICRO_SUBSET_N,
    "WORKSHOP_SUBSET_N": WORKSHOP_SUBSET_N,
    "INTERVAL_COVERAGE": INTERVAL_COVERAGE,
    "cv_params":         cv_params,
}

config_path = ARTIFACT_DIR / "02_global_config.pkl"
with open(config_path, "wb") as f:
    pickle.dump(global_config, f)

print(f"  ✓ Config artifact saved: {config_path.name}")

  ✓ Config artifact saved: 02_global_config.pkl


**Expected output:**
```
  ✓ Config artifact saved: 02_global_config.pkl
```